In [3]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix


class MyStandardScaler:

    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        X = np.array(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        # защита от деления на ноль
        self.std_[self.std_ == 0] = 1.0
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class MyKNN:
    """
    Параметры:
        k       — количество соседей
        metric  — 'euclidean' или 'manhattan'
        weights — 'uniform' (все соседи равны) или 'distance' (ближние важнее)
    """

    def __init__(self, k=5, metric="euclidean", weights="uniform"):
        self.k = k
        self.metric = metric
        self.weights = weights
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.array(X, dtype=float)
        self.y_train = np.array(y)
        return self

    def _distance(self, x1, x2):
        """Расстояние между двумя точками."""
        if self.metric == "euclidean":
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.metric == "manhattan":
            return np.sum(np.abs(x1 - x2))
        else:
            raise ValueError(f"Неизвестная метрика: {self.metric}")

    def _predict_one(self, x):
        """Класс для одной точки."""
        # 1. Расстояния до всех обучающих точек
        distances = np.array([self._distance(x, xt) for xt in self.X_train])

        # 2. Индексы k ближайших
        k_indices = np.argsort(distances)[: self.k]
        k_labels = self.y_train[k_indices]
        k_distances = distances[k_indices]

        # 3. Голосование
        if self.weights == "uniform":
            votes = Counter(k_labels)
        elif self.weights == "distance":
            votes = {}
            for label, dist in zip(k_labels, k_distances):
                votes[label] = votes.get(label, 0) + 1.0 / (dist + 1e-9)
        else:
            raise ValueError(f"Неизвестный тип весов: {self.weights}")

        # 4. Класс с максимумом голосов
        return max(votes, key=votes.get)

    def predict(self, X):
        X = np.array(X, dtype=float)
        return np.array([self._predict_one(x) for x in X])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.array(y))


def find_best_knn_params(X_train, y_train, X_test, y_test,
                         k_values=range(1, 16),
                         metrics=("euclidean", "manhattan"),
                         weights_options=("uniform", "distance")):
    """
    Ручной перебор гиперпараметров KNN.
    Возвращает лучшие (k, metric, weights), лучшую точность и таблицу результатов.
    """
    best = {"k": None, "metric": None, "weights": None, "acc": -1.0}
    results = []

    for metric in metrics:
        for weights in weights_options:
            for k in k_values:
                model = MyKNN(k=k, metric=metric, weights=weights)
                model.fit(X_train, y_train)
                acc = model.score(X_test, y_test)
                results.append((k, metric, weights, acc))
                if acc > best["acc"]:
                    best = {"k": k, "metric": metric,
                            "weights": weights, "acc": acc}

    return best, results


# ===============================================================
# 1. ЗАГРУЗКА
# ===============================================================
df = pd.read_csv(
    "Предпочтение пользователей Iphone или android.csv",
    encoding="utf-8-sig"
)
df.columns = [
    "timestamp", "stability", "photo_freq", "blog", "pay_for",
    "status_tech", "vpn", "age", "budget", "customization",
    "ringtone", "resale", "system_setup", "change_freq",
    "os_support", "sphere", "gender", "instagram", "charges_per_day",
    "save_or_treat", "headphones", "apples", "target"
]

# ===============================================================
# 2. ОЧИСТКА
# ===============================================================
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df[(df["age"] >= 14) & (df["age"] <= 80)]


def parse_budget(x):
    x = str(x).strip().replace(" ", "").replace(".", "").replace(",", ".")
    try:
        val = float(x)
    except ValueError:
        return np.nan
    return val * 1000 if val < 1000 else val


df["budget"] = df["budget"].apply(parse_budget)
df["change_freq"] = pd.to_numeric(
    df["change_freq"].astype(str).str.strip(), errors="coerce"
)
df["charges_per_day"] = pd.to_numeric(df["charges_per_day"], errors="coerce")
df["apples"] = pd.to_numeric(df["apples"], errors="coerce")

df = df.dropna(subset=["budget", "change_freq", "charges_per_day", "apples"])

df["target"] = df["target"].astype(str).str.strip().str.lower()
df = df[df["target"].isin(["айфон", "андроид"])].reset_index(drop=True)
df["target_bin"] = df["target"].map({"айфон": 1, "андроид": 0})

# ===============================================================
# 3. КОДИРОВАНИЕ
# ===============================================================
def split_multi(series):
    return series.fillna("").apply(
        lambda x: set(v.strip().lower()
                      for v in re.split(r"[;,]", str(x)) if v.strip())
    )


tags = set()
for s in split_multi(df["pay_for"]):
    tags.update(s)

for tag in sorted(tags):
    df[f"pay_for_{tag}"] = split_multi(df["pay_for"]).apply(
        lambda s: int(tag in s)
    )
df = df.drop(columns=["pay_for"])

cat_cols = [
    "stability", "photo_freq", "blog", "status_tech", "vpn",
    "ringtone", "resale", "system_setup", "os_support",
    "sphere", "gender", "instagram", "save_or_treat", "headphones"
]
encoders = {}
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

feature_cols = [c for c in df.columns
                if c not in ["timestamp", "target", "target_bin"]]
X = df[feature_cols].astype(float).values
y = df["target_bin"].astype(int).values

# ===============================================================
# 4. РАЗДЕЛЕНИЕ И ОБУЧЕНИЕ
# ===============================================================
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y
)

# Масштабирование своим классом
scaler = MyStandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Перебор гиперпараметров
best, all_results = find_best_knn_params(
    X_train_scaled, y_train, X_test_scaled, y_test
)

print("=" * 70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО (KNN написан вручную)")
print("=" * 70)
print(f"Лучшие параметры: k={best['k']}, "
      f"metric={best['metric']}, weights={best['weights']}")
print(f"Точность на тестовой выборке: {best['acc']:.1%}")
print(f"Обучающая выборка: {len(X_train)} чел. | "
      f"Тестовая: {len(X_test)} чел.")
print()

# Финальная модель с лучшими параметрами
model = MyKNN(k=best["k"], metric=best["metric"], weights=best["weights"])
model.fit(X_train_scaled, y_train)

# ===============================================================
# 5. ПОНЯТНЫЙ ВЫВОД ПО КАЖДОМУ ТЕСТУ
# ===============================================================
X_all_scaled = scaler.transform(X)
pred_all = model.predict(X_all_scaled)
pred_test = model.predict(X_test_scaled)


def fmt_sphere(code):
    """Числовой код сферы → исходный текст."""
    return encoders["sphere"].inverse_transform([int(code)])[0]


# --- 5.1. Вся выборка ---
print("=" * 70)
print("ПРЕДСКАЗАНИЯ ПО КАЖДОМУ РЕСПОНДЕНТУ (вся выборка)")
print("=" * 70)
print(f"{'№':>3} | {'Возраст':>7} | {'Бюджет':>9} | {'Сфера':<15} | "
      f"{'Предсказано':<11} | {'Реально':<10} | {'Итог'}")
print("-" * 75)

correct_total = 0
for i, (_, row) in enumerate(df.iterrows(), start=1):
    real = "Айфон" if row["target_bin"] == 1 else "Андроид"
    pred = "Айфон" if pred_all[i - 1] == 1 else "Андроид"
    ok = "+" if real == pred else "-"
    if real == pred:
        correct_total += 1
    sphere = fmt_sphere(row["sphere"])[:15]
    print(f"{i:>3} | {int(row['age']):>7} | {int(row['budget']):>9} | "
          f"{sphere:<15} | {pred:<11} | {real:<10} | {ok}")

print("-" * 75)
print(f"Итого угадано: {correct_total} из {len(df)} "
      f"({correct_total / len(df):.1%})")
print()

# --- 5.2. Тестовая выборка ---
print("=" * 70)
print("ПРЕДСКАЗАНИЯ ПО ТЕСТОВОЙ ВЫБОРКЕ (25% данных)")
print("=" * 70)
print(f"{'№':>3} | {'Возраст':>7} | {'Бюджет':>9} | {'Сфера':<15} | "
      f"{'Предсказано':<11} | {'Реально':<10} | {'Итог'}")
print("-" * 75)

correct_test = 0
for j, (idx, pred) in enumerate(zip(idx_test, pred_test), start=1):
    row = df.loc[idx]
    real = "Айфон" if row["target_bin"] == 1 else "Андроид"
    pred_lbl = "Айфон" if pred == 1 else "Андроид"
    ok = "+" if real == pred_lbl else "-"
    if real == pred_lbl:
        correct_test += 1
    sphere = fmt_sphere(row["sphere"])[:15]
    print(f"{j:>3} | {int(row['age']):>7} | {int(row['budget']):>9} | "
          f"{sphere:<15} | {pred_lbl:<11} | {real:<10} | {ok}")

print("-" * 75)
print(f"Итого угадано: {correct_test} из {len(X_test)} "
      f"({correct_test / len(X_test):.1%})")
print()

# ===============================================================
# 6. ОБЩИЙ РЕЗУЛЬТАТ
# ===============================================================
cm = confusion_matrix(y, pred_all)
tp, fn, fp, tn = cm[1, 1], cm[1, 0], cm[0, 1], cm[0, 0]

print("=" * 70)
print("ИТОГОВЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print(f"Всего опрошенных:        {len(df)}")
print(f"Айфонов в данных:        {(y == 1).sum()}")
print(f"Андроидов в данных:      {(y == 0).sum()}")
print()
print(f"Общая точность модели:   {accuracy_score(y, pred_all):.1%}")
print(f"Точность на тесте:       {accuracy_score(y_test, pred_test):.1%}")
print()
print("Матрица ошибок (вся выборка):")
print(f"  Правильно угадан Айфон:   {tp}")
print(f"  Айфон принят за Андроид:  {fn}")
print(f"  Андроид принят за Айфон:  {fp}")
print(f"  Правильно угадан Андроид: {tn}")
print()
print("Качество по классам:")
print(f"  Айфон  — точность {tp / (tp + fp):.1%}, "
      f"полнота {tp / (tp + fn):.1%}")
print(f"  Андроид — точность {tn / (tn + fn):.1%}, "
      f"полнота {tn / (tn + fp):.1%}")
print()


ОБУЧЕНИЕ ЗАВЕРШЕНО (KNN написан вручную)
Лучшие параметры: k=7, metric=euclidean, weights=uniform
Точность на тестовой выборке: 90.0%
Обучающая выборка: 30 чел. | Тестовая: 10 чел.

ПРЕДСКАЗАНИЯ ПО КАЖДОМУ РЕСПОНДЕНТУ (вся выборка)
  № | Возраст |    Бюджет | Сфера           | Предсказано | Реально    | Итог
---------------------------------------------------------------------------
  1 |      21 |    800000 | it              | Андроид     | Андроид    | +
  2 |      21 |    900000 | продажи         | Айфон       | Айфон      | +
  3 |      22 |    600000 | менеджер        | Андроид     | Айфон      | -
  4 |      22 |    500000 | студент         | Андроид     | Айфон      | -
  5 |      22 |    300000 | тех под         | Андроид     | Андроид    | +
  6 |      22 |    700000 | it              | Айфон       | Айфон      | +
  7 |      30 |    500000 | нету            | Айфон       | Айфон      | +
  8 |      21 |    700000 | инженер         | Айфон       | Айфон      | +
  9 |      22 